In [0]:
%run ./00_setup

In [0]:
# Generating Bin records

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

# --- 1. RESOURCE CALCULATION & CONSTRAINTS ---
available_bins_df = (spark.table("workspace.amazon_fullfilment_silver.bins")
                     .filter("status = 'idle' AND is_active = true"))

employees_count = (spark.table("workspace.amazon_fullfilment_silver.employee")
                   .filter(f"is_active = true AND job_role = 'Stower' AND shift_id = '{current_shift}'")
                   .count())

# Calculate capacity based on employees (1 employee : 5 bins)
employee_bin_capacity = employees_count * 5
# Total bins available in the warehouse (capped at 100 as per your requirement)
warehouse_bin_limit = min(available_bins_df.count(), 100)

# The final bottleneck constraint
max_bins_to_process = min(employee_bin_capacity, warehouse_bin_limit)

print(f"Capacity: {employees_count} employees can handle {employee_bin_capacity} bins. "
      f"Warehouse has {warehouse_bin_limit} idle bins. "
      f"Processing Limit: {max_bins_to_process} bins.")

# --- 2. DATA PREPARATION (The Join & Explode) ---
orders_in_picking = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
    .filter("is_current = true AND status = 'Picking'")
    .select("order_id"))

items_to_stow = (spark.table("workspace.amazon_fullfilment_silver.order_items_silver").alias("order_items")
    .join(orders_in_picking, "order_id")
    .join(spark.table("workspace.amazon_fullfilment_silver.products_silver").filter("is_current=true"), "product_id")
    .withColumn("quantity_array", F.expr("explode(sequence(1, quantity))"))
    .withColumn("unique_item_id", F.monotonically_increasing_id())
    .select("unique_item_id", "order_items.order_id", "weight_kg"))

# --- 3. VIRTUAL BINNING ---
strict_window = Window.orderBy("unique_item_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)

bin_calculation = (items_to_stow
    .withColumn("running_total", F.sum("weight_kg").over(strict_window))
    .withColumn("virtual_bin_id", F.floor((F.col("running_total") - F.lit(0.001)) / 60.0)))

# --- 4. APPLY THE WORK LIMIT ---
# We group and then limit the number of virtual bins based on our constraint
virtual_bins_final = (bin_calculation.groupBy("virtual_bin_id")
    .agg(
        F.sum("weight_kg").alias("bin_total_weight"),
        # Change: Convert array of IDs to a comma-separated string for CSV compatibility
        F.concat_ws(", ", F.collect_set("order_id")).alias("formatted_order_ids")
    )
    .orderBy("virtual_bin_id")
    .limit(max_bins_to_process)) # APPLY EMPLOYEE/WAREHOUSE CONSTRAINT HERE

# --- 5. PHYSICAL MAPPING & OUTPUT ---
available_bins_ranked = available_bins_df.withColumn("bin_rank", F.row_number().over(Window.orderBy("bin_id")))
ranked_virtual_bins = virtual_bins_final.withColumn("v_rank", F.row_number().over(Window.orderBy("virtual_bin_id")))

bin_assignments_df = (ranked_virtual_bins.join(available_bins_ranked, ranked_virtual_bins.v_rank == available_bins_ranked.bin_rank)
    .withColumn("status", 
        F.when((F.col("bin_total_weight") > 50) , "Shipment")
        .otherwise("Picking"))
    .select(
        "bin_id",
        F.col("formatted_order_ids").alias("order_id"),
        F.lit(0).alias("item_count"),
         "status",  
        F.col("bin_total_weight").cast("double").alias("current_weight"), 
        F.lit("N/A").alias("employee_id"),
        F.current_timestamp().alias("updated_timestamp"),
        F.lit(current_run_uuid).alias("_batch_id"),
        F.current_timestamp().alias("_ingest_timestamp")
    ))

#display(bin_assignments_df)

(bin_assignments_df
 .drop("_metadata")
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{source_volume_path}/bin_events"))